## Set up

### libraries

In [53]:
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from collections import defaultdict
from openpyxl.utils import get_column_letter


### Data Extraction

In [54]:
def load_lift_drag_data(root_dir):
    data_by_config_aoa = defaultdict(lambda: {'lift': [], 'drag': [], 'turbulence_model': '', 'geometry': '', 'mesh': '', 'version': ''})
    
    for dirpath, _, filenames in os.walk(root_dir):
        # Find lift and drag files
        lift_files = [f for f in filenames if 'lift_force' in f and f.endswith('.txt')]
        drag_files = [f for f in filenames if 'drag_force' in f and f.endswith('.txt')]
        
        if lift_files and drag_files:
            config = None
            
            # Extract configuration based on method
            if CONFIG_EXTRACTION_METHOD == 'case_file':
                # Look for .cas or .cas.h5 files and extract filename
                for filename in filenames:
                    if filename.endswith('.cas') or filename.endswith('.cas.h5'):
                        config = filename.replace('.cas.h5', '').replace('.cas', '')
                        if config and config[0].isdigit() and '.' in config:
                            break
                        else:
                            config = None
            else:
                # Legacy: parse from folder structure
                parts = dirpath.split(os.sep)
                for part in parts:
                    if part and part[0].isdigit() and part.count('.') >= 2:
                        config = part
                        break
            
            # Find AoA value from folder name (e.g., AoA_10)
            aoa = None
            parts = dirpath.split(os.sep)
            for part in parts:
                if part.startswith('AoA_'):
                    aoa = part
                    break
            
            if config and aoa:
                # Extract AoA number
                aoa_number = aoa.split('_')[1]
                
                # Check if config contains _AoA_ pattern (old format embedded in filename)
                # Example: "4.3.1.3_AoA_5" becomes "4.3.1.3.5"
                if '_AoA_' in config:
                    config = config.replace(f'_AoA_{aoa_number}', f'.{aoa_number}')
                # Check if this is old format with no dots (version name only)
                # Example: "version_name" becomes "version_name.5"
                elif config.count('.') == 0:
                    config = f"{config}.{aoa_number}"
                
                # Parse configuration using position and value mappings
                config_parts = config.split('.')
                positions = POSITION_MAP
                mappings = VALUE_MAPPINGS
                
                # Extract each field based on position
                geometry_num = config_parts[positions['geometry']] if positions['geometry'] is not None and len(config_parts) > positions['geometry'] else None
                mesh_num = config_parts[positions['mesh']] if positions['mesh'] is not None and len(config_parts) > positions['mesh'] else None
                turbulence_num = config_parts[positions['turbulence']] if positions['turbulence'] is not None and len(config_parts) > positions['turbulence'] else None
                version_num = config_parts[positions['version']] if positions['version'] is not None and len(config_parts) > positions['version'] else None
                
                # Map to descriptive names
                geometry = mappings.get('geometry', {}).get(geometry_num, f"Geometry_{geometry_num}") if geometry_num else "N/A"
                mesh = mappings.get('mesh', {}).get(mesh_num, f"Mesh_{mesh_num}") if mesh_num else "N/A"
                turbulence_model = mappings.get('turbulence', {}).get(turbulence_num, f"Turbulence_{turbulence_num}") if turbulence_num else "Unknown"
                version = mappings.get('version', {}).get(version_num, f"Version_{version_num}") if version_num else "N/A"
                
                # Extract angle of attack in degrees (e.g., AoA_10 → 10)
                aoa_degrees = float(aoa_number)
                aoa_radians = np.radians(aoa_degrees)
                
                # Read all lift files
                for lift_file in lift_files:
                    lift_path = os.path.join(dirpath, lift_file)
                    with open(lift_path) as f:
                        lift_data = []
                        for line in f:
                            line = line.strip()
                            # Skip header lines (lines with quotes or non-numeric content)
                            if not line or '"' in line or '(' in line:
                                continue
                            # Split by whitespace and take the second column
                            parts_line = line.split()
                            if len(parts_line) >= 2:
                                try:
                                    value = float(parts_line[1])
                                    lift_data.append(value)
                                except ValueError:
                                    continue
                
                # Read all drag files
                for drag_file in drag_files:
                    drag_path = os.path.join(dirpath, drag_file)
                    with open(drag_path) as f:
                        drag_data = []
                        for line in f:
                            line = line.strip()
                            # Skip header lines (lines with quotes or non-numeric content)
                            if not line or '"' in line or '(' in line:
                                continue
                            # Split by whitespace and take the second column
                            parts_line = line.split()
                            if len(parts_line) >= 2:
                                try:
                                    value = float(parts_line[1])
                                    drag_data.append(value)
                                except ValueError:
                                    continue
                
                # Apply angle of attack correction to transform forces
                # True lift = lift*cos(theta) - drag*sin(theta)
                # True drag = lift*sin(theta) + drag*cos(theta)
                cos_theta = np.cos(aoa_radians)
                sin_theta = np.sin(aoa_radians)
                
                true_lift_data = []
                true_drag_data = []
                
                # Ensure both arrays have the same length
                min_length = min(len(lift_data), len(drag_data))
                for i in range(min_length):
                    true_lift = lift_data[i] * cos_theta - drag_data[i] * sin_theta
                    true_drag = lift_data[i] * sin_theta + drag_data[i] * cos_theta
                    true_lift_data.append(true_lift)
                    true_drag_data.append(true_drag)
                
                # Store the corrected data
                data_by_config_aoa[(config, aoa)]['lift'].extend(true_lift_data)
                data_by_config_aoa[(config, aoa)]['drag'].extend(true_drag_data)
                
                # Store metadata
                data_by_config_aoa[(config, aoa)]['geometry'] = geometry
                data_by_config_aoa[(config, aoa)]['mesh'] = mesh
                data_by_config_aoa[(config, aoa)]['turbulence_model'] = turbulence_model
                data_by_config_aoa[(config, aoa)]['version'] = version
    
    return data_by_config_aoa


In [55]:
def compute_statistics(data):
    mean_val = np.mean(data)
    std_dev = np.std(data)
    # Calculate Coefficient of Variation (COV) as percentage
    cov = (std_dev / mean_val * 100) if mean_val != 0 else 0
    return mean_val, cov

## Enter Values Here

In [56]:
# ==================== CONFIGURATION ====================
# User parameters - modify these as needed

base_path = r"C:\Users\lukek\OneDrive - Liberty University\Group-F.L.U.I.D. Research - GRID-FINS - GRID-FINS\Test\2414_006_004.3"
output_dir = r"C:\Users\lukek\OneDrive - Liberty University\Group-F.L.U.I.D. Research - GRID-FINS - GRID-FINS\Test\2414_006_004.3"

# ==================== COEFFICIENT CALCULATION PARAMETERS ====================
# Geometry parameters
SPAN = .85344    # Wingspan in meters
CHORD = .1  # Chord length in meters

# Calculate planform area
REFERENCE_AREA = SPAN * CHORD  # Planform area in m² (span × chord)

# Flow conditions
AIR_DENSITY = 1.225    # Air density in kg/m³ (standard sea level)
VELOCITY = 24.38        # Freestream velocity in m/s

# Calculate dynamic pressure
DYNAMIC_PRESSURE = 0.5 * AIR_DENSITY * VELOCITY**2  # Pa
Q_times_A = DYNAMIC_PRESSURE * REFERENCE_AREA  # N

# ==================== COEFFICIENT GRAPH FILTER ====================
# Select which turbulence model to graph for coefficients vs AoA
# Options: 'SST', 'RNG', 'RSM', 'k-epsilon', or 'ALL' to graph all models
COEFF_TURBULENCE_MODEL = 'SST'  # Change this to select which model to graph

# Select which version to graph for coefficients vs AoA
# *** UPDATE VERSION OPTIONS HERE WHEN VERSION NAMES CHANGE ***
# Options: 'Version 1', 'Version 2', 'Version 3', 'Version 4', or 'ALL' to graph all versions
# Make sure this matches the version names in VALUE_MAPPINGS['version'] below
COEFF_VERSION = 'Version 2'  # Change this to select which version to graph

num_iterations = 150  # Number of latest iterations to use for statistics

# Select configuration extraction method:
# 'case_file'   = Read config directly from .cas/.cas.h5 filename (RECOMMENDED)
# 'folder_path' = Parse config from folder structure (legacy method)
CONFIG_EXTRACTION_METHOD = 'case_file'

# ==================== POSITION MAPPING ====================
# Tell the code which position in the config corresponds to which field
# Example: For config "4.3.1.2.5", specify which number is mesh, turbulence, AoA, etc.
POSITION_MAP = {
    'geometry': 0,      # Position 0 = first number (4 in "4.3.1.2.5")
    'mesh': 1,          # Position 1 = second number (3 in "4.3.1.2.5")
    'turbulence': 2,    # Position 2 = third number (1 in "4.3.1.2.5")
    'version': 3,       # Position 3 = fourth number (2 in "4.3.1.2.5")
    'aoa': 4,           # Position 4 = fifth number (5 in "4.3.1.2.5") - NEW FORMAT ONLY
    'status': None      # Position for status field (or None if not present)
}

# ==================== VALUE MAPPINGS ====================
# Define what each number means for each position
VALUE_MAPPINGS = {
    'geometry': {
        '1': 'Baseline Geometry',
        '2': 'Modified Geometry',
        '3': 'Grid Fins Geometry',
        '4': 'Baseline'
    },
    'mesh': {
        '1': 'Coarse Mesh',
        '2': 'Medium Mesh',
        '3': 'Standard',
        '4': 'NM Adjusted'
    },
    'turbulence': {
        '1': 'SST',
        '2': 'RNG',
        '3': 'RSM',
        '4': 'k-epsilon'
    },
    'version': {
        # *** UPDATE VERSION NAMES HERE ***
        # When version names change, update both the names below AND the COEFF_VERSION parameter above
        '1': 'Version 1',
        '2': 'Version 2',
        '3': 'Version 3',
        '4': 'Version 4'
        # Add more versions as needed: '5': 'Version 5', etc.
    },
    'aoa': {
        '0': '0 degrees',
        '1': '1 degrees',
        '2': '2 degrees',
        '3': '3 degrees',
        '4': '4 degrees',
        '5': '5 degrees',
        '6': '6 degrees',
        '7': '7 degrees',
        '8': '8 degrees',
        '9': '9 degrees',
        '10': '10 degrees',
        '11': '11 degrees',
        '12': '12 degrees',
        '13': '13 degrees',
        '14': '14 degrees',
        '15': '15 degrees',
        '16': '16 degrees',
        '17': '17 degrees',
        '18': '18 degrees',
        '19': '19 degrees',
        '20': '20 degrees',
        '21': '21 degrees',
        '22': '22 degrees',
        '23': '23 degrees',
        '24': '24 degrees',
        '25': '25 degrees'
    }
}

# Display current configuration
print("=" * 80)
print("CONFIGURATION SETTINGS:")
print("=" * 80)
print(f"✓ Data Path: {base_path}")
print(f"✓ Output Directory: {output_dir}")
print(f"✓ Iterations for Statistics: {num_iterations}")
print(f"✓ Config Extraction: {CONFIG_EXTRACTION_METHOD}")
if CONFIG_EXTRACTION_METHOD == 'case_file':
    print(f"  → Reading configuration from .cas file")
    print(f"  → Position mapping: geometry={POSITION_MAP['geometry']}, mesh={POSITION_MAP['mesh']}, turbulence={POSITION_MAP['turbulence']}, version={POSITION_MAP['version']}, aoa={POSITION_MAP['aoa']}")
else:
    print(f"  → Parsing from folder structure (legacy mode)")
print("\nCOEFFICIENT PARAMETERS:")
print(f"✓ Span: {SPAN} m")
print(f"✓ Chord: {CHORD} m")
print(f"✓ Reference Area: {REFERENCE_AREA:.6f} m²")
print(f"✓ Air Density: {AIR_DENSITY} kg/m³")
print(f"✓ Velocity: {VELOCITY} m/s")
print(f"✓ Dynamic Pressure (q): {DYNAMIC_PRESSURE:.2f} Pa")
print(f"✓ q×A: {Q_times_A:.6f} N")
print("=" * 80)

# ==================== PROCESSING ====================

# Load all data
all_data = load_lift_drag_data(base_path)

# Create output directories
os.makedirs(output_dir, exist_ok=True)
raw_data_dir = os.path.join(output_dir, "raw_data")
os.makedirs(raw_data_dir, exist_ok=True)

# Sort AoA values numerically (5, 10, 15, 20)
def extract_aoa_number(aoa_str):
    return int(aoa_str.split('_')[1])

# Export each dataset with a unique name based on config only (config already includes AoA)
for (config, aoa), data in all_data.items():
    # Use config as filename base since it already includes the AoA (e.g., "4.3.1.3.5")
    filename_base = config
    
    # Export lift data to raw_data folder
    lift_output = os.path.join(raw_data_dir, f"{filename_base}_lift.txt")
    with open(lift_output, 'w') as f:
        for value in data['lift']:
            f.write(f"{value}\n")
    
    # Export drag data to raw_data folder
    drag_output = os.path.join(raw_data_dir, f"{filename_base}_drag.txt")
    with open(drag_output, 'w') as f:
        for value in data['drag']:
            f.write(f"{value}\n")
    
    print(f"Exported: {filename_base}")

# Create summary statistics file using specified number of iterations
summary_file = os.path.join(output_dir, "SUMMARY_Statistics.txt")

with open(summary_file, 'w') as f:
    f.write("=" * 100 + "\n")
    f.write(f"SUMMARY STATISTICS - Last {num_iterations} Iterations\n")
    f.write("=" * 100 + "\n\n")
    
    # Sort by config and then by AoA numerically
    sorted_data = sorted(all_data.items(), key=lambda x: (x[0][0], extract_aoa_number(x[0][1])))
    
    for (config, aoa), data in sorted_data:
        # Get last N iterations (or all if less than N)
        lift_last_n = data['lift'][-num_iterations:] if len(data['lift']) >= num_iterations else data['lift']
        drag_last_n = data['drag'][-num_iterations:] if len(data['drag']) >= num_iterations else data['drag']
        
        # Calculate statistics
        lift_mean = np.mean(lift_last_n) if lift_last_n else 0
        lift_std = np.std(lift_last_n) if lift_last_n else 0
        lift_cov = (lift_std / lift_mean * 100) if lift_mean != 0 else 0
        
        drag_mean = np.mean(drag_last_n) if drag_last_n else 0
        drag_std = np.std(drag_last_n) if drag_last_n else 0
        drag_cov = (drag_std / drag_mean * 100) if drag_mean != 0 else 0
        
        # Write to file
        f.write(f"Configuration: {config}\n")
        f.write(f"Turbulence Model: {data['turbulence_model']}\n")
        f.write(f"Version: {data['version']}\n")
        f.write(f"Angle of Attack: {aoa}\n")
        f.write(f"Iterations used: {len(lift_last_n)}\n\n")
        f.write(f"LIFT:\n")
        f.write(f"  Average: {lift_mean:12.6f}\n")
        f.write(f"  COV (%): {lift_cov:12.6f}\n\n")
        f.write(f"DRAG:\n")
        f.write(f"  Average: {drag_mean:12.6f}\n")
        f.write(f"  COV (%): {drag_cov:12.6f}\n")
        f.write("-" * 100 + "\n\n")

CONFIGURATION SETTINGS:
✓ Data Path: C:\Users\lukek\OneDrive - Liberty University\Group-F.L.U.I.D. Research - GRID-FINS - GRID-FINS\Test\2414_006_004.3
✓ Output Directory: C:\Users\lukek\OneDrive - Liberty University\Group-F.L.U.I.D. Research - GRID-FINS - GRID-FINS\Test\2414_006_004.3
✓ Iterations for Statistics: 150
✓ Config Extraction: case_file
  → Reading configuration from .cas file
  → Position mapping: geometry=0, mesh=1, turbulence=2, version=3, aoa=4

COEFFICIENT PARAMETERS:
✓ Span: 0.85344 m
✓ Chord: 0.1 m
✓ Reference Area: 0.085344 m²
✓ Air Density: 1.225 kg/m³
✓ Velocity: 24.38 m/s
✓ Dynamic Pressure (q): 364.06 Pa
✓ q×A: 31.070375 N
Exported: 4.3.1.2.10
Exported: 4.3.1.2.15
Exported: 4.3.1.2.20
Exported: 4.3.1.2.5
Exported: 4.3.1.3.10
Exported: 4.3.1.3.15
Exported: 4.3.1.3.20
Exported: 4.3.1.3.5
Exported: 4.3.1.4.10
Exported: 4.3.1.4.15
Exported: 4.3.1.4.20
Exported: 4.3.1.4.5
Exported: 4.3.2.1.10
Exported: 4.3.2.1.15
Exported: 4.3.2.1.5
Exported: 4.3.3.1.10
Exported: 4.3

In [57]:
# ==================== CREATE EXCEL TABLES ====================

from openpyxl.utils import get_column_letter
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

# Create Excel file with all data
excel_file = os.path.join(output_dir, "SUMMARY_Statistics.xlsx")

# Recreate config_groups for reference by turbulence model
config_groups = defaultdict(lambda: {'lift': {}, 'drag': {}})
for (config, aoa), data in all_data.items():
    config_parts = config.split('.')
    # Get turbulence number from position specified in POSITION_MAP
    turbulence_pos = POSITION_MAP['turbulence']
    turbulence_num = config_parts[turbulence_pos] if turbulence_pos is not None and len(config_parts) > turbulence_pos else "unknown"
    
    lift_last_n = data['lift'][-num_iterations:] if len(data['lift']) >= num_iterations else data['lift']
    drag_last_n = data['drag'][-num_iterations:] if len(data['drag']) >= num_iterations else data['drag']
    
    config_groups[turbulence_num]['lift'][aoa] = lift_last_n
    config_groups[turbulence_num]['drag'][aoa] = drag_last_n

with pd.ExcelWriter(excel_file, engine='openpyxl') as writer:
    # Create a summary sheet with all configurations
    summary_data = []
    
    # Sort by config and then by AoA numerically
    sorted_data = sorted(all_data.items(), key=lambda x: (x[0][0], extract_aoa_number(x[0][1])))
    
    for (config, aoa), data in sorted_data:
        # Get last N iterations
        lift_last_n = data['lift'][-num_iterations:] if len(data['lift']) >= num_iterations else data['lift']
        drag_last_n = data['drag'][-num_iterations:] if len(data['drag']) >= num_iterations else data['drag']
        
        # Calculate statistics
        lift_mean = np.mean(lift_last_n) if lift_last_n else 0
        lift_std = np.std(lift_last_n) if lift_last_n else 0
        lift_cov = (lift_std / lift_mean * 100) if lift_mean != 0 else 0
        
        drag_mean = np.mean(drag_last_n) if drag_last_n else 0
        drag_std = np.std(drag_last_n) if drag_last_n else 0
        drag_cov = (drag_std / drag_mean * 100) if drag_mean != 0 else 0
        
        summary_data.append({
            'Configuration': config,
            'Turbulence Model': data['turbulence_model'],
            'Version': data['version'],
            'Angle of Attack': aoa,
            'Iterations Used': len(lift_last_n),
            'Lift Mean (N)': f"{lift_mean:.3f}",
            'Lift COV (%)': f"{lift_cov:.2f}",
            'Drag Mean (N)': f"{drag_mean:.3f}",
            'Drag COV (%)': f"{drag_cov:.2f}"
        })
    
    # Write summary to main sheet
    summary_df = pd.DataFrame(summary_data)
    summary_df.to_excel(writer, sheet_name='Summary', index=False)
    
    # Create consolidated Turbulence Models sheet
    turbulence_models_data = []
    
    for turbulence_num in sorted(config_groups.keys()):
        # Sort AoA numerically within each turbulence model type
        sorted_aoas = sorted(config_groups[turbulence_num]['lift'].keys(), key=extract_aoa_number)
        
        for aoa in sorted_aoas:
            lift_data = config_groups[turbulence_num]['lift'][aoa]
            drag_data = config_groups[turbulence_num]['drag'][aoa]
            
            # Get turbulence model info (should be same for all in this group)
            turbulence_model = ""
            version = ""
            for (config, config_aoa), d in all_data.items():
                if config_aoa == aoa:
                    config_parts = config.split('.')
                    turb_pos = POSITION_MAP['turbulence']
                    if turb_pos is not None and len(config_parts) > turb_pos and config_parts[turb_pos] == turbulence_num:
                        turbulence_model = d['turbulence_model']
                        version = d['version']
                        break
            
            # Calculate COV for lift and drag
            lift_mean_val = np.mean(lift_data)
            lift_std_val = np.std(lift_data)
            lift_cov_val = (lift_std_val / lift_mean_val * 100) if lift_mean_val != 0 else 0
            
            drag_mean_val = np.mean(drag_data)
            drag_std_val = np.std(drag_data)
            drag_cov_val = (drag_std_val / drag_mean_val * 100) if drag_mean_val != 0 else 0
            
            turbulence_models_data.append({
                'Turbulence Model': turbulence_model,
                'Version': version,
                'Angle of Attack': aoa,
                'Lift Mean (N)': f"{lift_mean_val:.3f}",
                'Lift COV (%)': f"{lift_cov_val:.2f}",
                'Drag Mean (N)': f"{drag_mean_val:.3f}",
                'Drag COV (%)': f"{drag_cov_val:.2f}",
                'Num Points': len(lift_data)
            })
    
    turbulence_models_df = pd.DataFrame(turbulence_models_data)
    turbulence_models_df.to_excel(writer, sheet_name='Turbulence_Models', index=False)

# Apply formatting to all sheets
from openpyxl import load_workbook
wb = load_workbook(excel_file)

# Define styles
header_font = Font(name='Calibri', size=11, bold=True, color='FFFFFF')
header_fill = PatternFill(start_color='366092', end_color='366092', fill_type='solid')
header_alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)

data_alignment = Alignment(horizontal='center', vertical='center')
border_style = Border(
    left=Side(style='thin', color='000000'),
    right=Side(style='thin', color='000000'),
    top=Side(style='thin', color='000000'),
    bottom=Side(style='thin', color='000000')
)

# Alternating row colors
row_fill_light = PatternFill(start_color='F2F2F2', end_color='F2F2F2', fill_type='solid')
row_fill_white = PatternFill(start_color='FFFFFF', end_color='FFFFFF', fill_type='solid')

for ws_name in wb.sheetnames:
    worksheet = wb[ws_name]
    
    # Autofit column widths
    for column in worksheet.columns:
        max_length = 0
        column_letter = get_column_letter(column[0].column)
        for cell in column:
            try:
                if len(str(cell.value)) > max_length:
                    max_length = len(str(cell.value))
            except:
                pass
        adjusted_width = min(max_length + 2, 50)
        worksheet.column_dimensions[column_letter].width = adjusted_width
    
    # Format header row
    for cell in worksheet[1]:
        cell.font = header_font
        cell.fill = header_fill
        cell.alignment = header_alignment
        cell.border = border_style
    
    # Format data rows with alternating colors and borders
    for row_idx, row in enumerate(worksheet.iter_rows(min_row=2), start=2):
        fill_color = row_fill_light if row_idx % 2 == 0 else row_fill_white
        for cell in row:
            cell.alignment = data_alignment
            cell.border = border_style
            cell.fill = fill_color
    
    # Set row height for header
    worksheet.row_dimensions[1].height = 30

wb.save(excel_file)

print(f"✓ Excel file created: {excel_file}")
print(f"✓ Sheet names: Summary, Turbulence_Models")
print(f"✓ Professional formatting applied: colored headers, borders, alternating rows")
print(f"✓ All lift and drag values displayed in Newtons (N)")

✓ Excel file created: C:\Users\lukek\OneDrive - Liberty University\Group-F.L.U.I.D. Research - GRID-FINS - GRID-FINS\Test\2414_006_004.3\SUMMARY_Statistics.xlsx
✓ Sheet names: Summary, Turbulence_Models
✓ Professional formatting applied: colored headers, borders, alternating rows
✓ All lift and drag values displayed in Newtons (N)


In [ ]:
# ==================== CREATE TURBULENCE MODEL COMPARISON TABLE ====================

import time

# Create a comparison table for turbulence models
turbulence_comparison = []

# Get all unique AoAs
all_aoas = set()
for (config, aoa), data in all_data.items():
    all_aoas.add(aoa)

# Sort AoAs numerically
sorted_aoas = sorted(all_aoas, key=extract_aoa_number)

# Dynamically determine which turbulence models are available
all_models = set()
for (config, aoa), data in all_data.items():
    all_models.add(data['turbulence_model'])

print(f"✓ Turbulence models in dataset: {sorted(all_models)}")

# For each AoA, get data for each turbulence model
for aoa in sorted_aoas:
    aoa_entry = {'Angle of Attack': aoa}
    
    # Collect lift and drag means and COV for each model
    lift_means = {}
    drag_means = {}
    lift_covs = {}
    drag_covs = {}
    
    for model in sorted(all_models):
        model_lift_values = []
        model_drag_values = []
        
        # Find all configs with this AOA and turbulence model
        for (config, config_aoa), data in all_data.items():
            if config_aoa == aoa and data['turbulence_model'] == model:
                lift_last_n = data['lift'][-num_iterations:] if len(data['lift']) >= num_iterations else data['lift']
                drag_last_n = data['drag'][-num_iterations:] if len(data['drag']) >= num_iterations else data['drag']
                
                if lift_last_n:
                    model_lift_values.extend(lift_last_n)
                if drag_last_n:
                    model_drag_values.extend(drag_last_n)
        
        # Calculate means and COV for this model
        if model_lift_values:
            lift_means[model] = np.mean(model_lift_values)
            lift_std = np.std(model_lift_values)
            lift_covs[model] = (lift_std / lift_means[model] * 100) if lift_means[model] != 0 else 0
        else:
            lift_means[model] = None
            lift_covs[model] = None
        
        if model_drag_values:
            drag_means[model] = np.mean(model_drag_values)
            drag_std = np.std(model_drag_values)
            drag_covs[model] = (drag_std / drag_means[model] * 100) if drag_means[model] != 0 else 0
        else:
            drag_means[model] = None
            drag_covs[model] = None
    
    # Add lift values with COV and percent difference from SST
    lift_str = ""
    sst_lift = lift_means.get('SST')
    # Sort models by their numeric mapping (smallest to largest)
    model_order = {'SST': 1, 'RNG': 2, 'RSM': 3, 'k-epsilon': 4}
    sorted_models = sorted(all_models, key=lambda x: model_order.get(x, 99))
    for model in sorted_models:
        if lift_means[model] is not None and lift_covs[model] is not None:
            if lift_str:
                lift_str += " | "
            # Calculate percent difference from SST
            if model != 'SST' and sst_lift is not None and sst_lift != 0:
                pct_diff = ((lift_means[model] - sst_lift) / sst_lift * 100)
                lift_str += f"{model}: {lift_means[model]:.3f} N ({lift_covs[model]:.2f}%, {pct_diff:+.2f}% vs SST)"
            else:
                lift_str += f"{model}: {lift_means[model]:.3f} N ({lift_covs[model]:.2f}%)"
        else:
            if lift_str:
                lift_str += " | "
            lift_str += f"{model}: N/A"
    aoa_entry['Lift (N)'] = lift_str
    
    # Add drag values with COV and percent difference from SST
    drag_str = ""
    sst_drag = drag_means.get('SST')
    for model in sorted_models:
        if drag_means[model] is not None and drag_covs[model] is not None:
            if drag_str:
                drag_str += " | "
            # Calculate percent difference from SST
            if model != 'SST' and sst_drag is not None and sst_drag != 0:
                pct_diff = ((drag_means[model] - sst_drag) / sst_drag * 100)
                drag_str += f"{model}: {drag_means[model]:.3f} N ({drag_covs[model]:.2f}%, {pct_diff:+.2f}% vs SST)"
            else:
                drag_str += f"{model}: {drag_means[model]:.3f} N ({drag_covs[model]:.2f}%)"
        else:
            if drag_str:
                drag_str += " | "
            drag_str += f"{model}: N/A"
    aoa_entry['Drag (N)'] = drag_str
    
    turbulence_comparison.append(aoa_entry)

# Add to existing Excel file
comparison_df = pd.DataFrame(turbulence_comparison)

# Write comparison sheet
with pd.ExcelWriter(excel_file, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    comparison_df.to_excel(writer, sheet_name='Turbulence_Model_Comparison', index=False)

# Apply formatting to Turbulence_Model_Comparison sheet
time.sleep(0.5)  # Brief delay to ensure file is fully released

from openpyxl.styles.colors import Color
from openpyxl.cell.text import InlineFont
from openpyxl.cell.rich_text import TextBlock, CellRichText

wb = load_workbook(excel_file)
ws = wb['Turbulence_Model_Comparison']

# Define styles
header_font = Font(name='Calibri', size=11, bold=True, color='FFFFFF')
header_fill = PatternFill(start_color='366092', end_color='366092', fill_type='solid')
header_alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
data_alignment = Alignment(horizontal='left', vertical='center', wrap_text=True)
border_style = Border(
    left=Side(style='thin', color='000000'),
    right=Side(style='thin', color='000000'),
    top=Side(style='thin', color='000000'),
    bottom=Side(style='thin', color='000000')
)
row_fill_light = PatternFill(start_color='F2F2F2', end_color='F2F2F2', fill_type='solid')
row_fill_white = PatternFill(start_color='FFFFFF', end_color='FFFFFF', fill_type='solid')

# Color coding for highlights (use string format for InlineFont)
cov_color = 'FF0000'  # Red for COV
pct_diff_color = '4169E1'  # Royal blue for percentage comparison

# Autofit column widths
for column in ws.columns:
    max_length = 0
    column_letter = get_column_letter(column[0].column)
    for cell in column:
        try:
            if len(str(cell.value)) > max_length:
                max_length = len(str(cell.value))
        except:
            pass
    adjusted_width = min(max_length + 2, 80)
    ws.column_dimensions[column_letter].width = adjusted_width

# Format header row with legend
for cell in ws[1]:
    cell.font = header_font
    cell.fill = header_fill
    cell.alignment = header_alignment
    cell.border = border_style

# Add legend row below header to explain format
ws.insert_rows(2)
legend_font = Font(name='Calibri', size=9, italic=True)
legend_fill = PatternFill(start_color='E0E0E0', end_color='E0E0E0', fill_type='solid')

ws['A2'] = 'Legend'
ws['B2'] = 'Format: Model: Mean N (COV%, %Diff vs SST) | Red=COV | Blue=%Difference'
ws['A2'].font = legend_font
ws['B2'].font = legend_font
ws['A2'].fill = legend_fill
ws['B2'].fill = legend_fill
ws['A2'].border = border_style
ws['B2'].border = border_style
ws['A2'].alignment = Alignment(horizontal='center', vertical='center')
ws['B2'].alignment = Alignment(horizontal='left', vertical='center')
ws.merge_cells('B2:C2')

# Format data rows with alternating colors, borders, and highlighting
for row_idx, row in enumerate(ws.iter_rows(min_row=3), start=3):
    fill_color = row_fill_light if row_idx % 2 == 1 else row_fill_white
    for cell_idx, cell in enumerate(row):
        cell.alignment = data_alignment
        cell.border = border_style
        cell.fill = fill_color
        
        # Apply color highlighting to Lift and Drag columns (columns B and C)
        if cell_idx in [1, 2] and cell.value:  # Columns B and C (Lift and Drag)
            cell_text = str(cell.value)
            from openpyxl.cell.text import InlineFont
            from openpyxl.cell.rich_text import TextBlock, CellRichText
            
            # Parse and create rich text with colored segments
            # Format is: "Model: Value N (COV%, +X% vs SST)" or "Model: Value N (COV%)"
            text_parts = []
            i = 0
            while i < len(cell_text):
                # Check for opening parenthesis
                if cell_text[i] == '(':
                    # Find the closing parenthesis
                    closing_paren = cell_text.find(')', i)
                    if closing_paren != -1:
                        segment = cell_text[i:closing_paren+1]
                        
                        # Check if this segment contains "vs SST"
                        if 'vs SST' in segment:
                            # This segment has both COV and percentage difference
                            # Split by comma to separate them
                            if ',' in segment:
                                parts = segment[1:-1].split(',', 1)  # Remove outer parentheses and split once
                                # First part is COV (e.g., "3.45%")
                                cov_part = f"({parts[0].strip()})"
                                text_parts.append(TextBlock(InlineFont(color=cov_color), cov_part))
                                # Second part is percentage diff (e.g., " +1.96% vs SST")
                                pct_part = f"({parts[1].strip()})"
                                text_parts.append(TextBlock(InlineFont(color=pct_diff_color), pct_part))
                            else:
                                # Shouldn't happen, but just in case
                                text_parts.append(TextBlock(InlineFont(color=pct_diff_color), segment))
                        elif '%' in segment:
                            # This is just COV percentage - color it red
                            text_parts.append(TextBlock(InlineFont(color=cov_color), segment))
                        else:
                            # No percentage, regular text
                            text_parts.append(TextBlock(InlineFont(), segment))
                        
                        i = closing_paren + 1
                        continue
                
                # Regular text (not in parentheses)
                j = i
                while j < len(cell_text) and cell_text[j] != '(':
                    j += 1
                if j > i:
                    text_parts.append(TextBlock(InlineFont(), cell_text[i:j]))
                i = j
            
            if text_parts:
                cell.value = CellRichText(*text_parts)

# Set row height for header and legend
ws.row_dimensions[1].height = 30
ws.row_dimensions[2].height = 25
wb.save(excel_file)
wb.close()

print(f"\n✓ Turbulence Model Comparison table added to Excel file")
print(f"✓ Format: Model: Mean (COV%) with percent difference from SST")
print(f"✓ Professional formatting applied to comparison sheet")
print(f"\nSample comparison:")
for row in turbulence_comparison:
    print(f"\n{row['Angle of Attack']}:")
    print(f"  Lift (N):  {row['Lift (N)']}")
    print(f"  Drag (N):  {row['Drag (N)']}")

✓ Turbulence models in dataset: ['RNG', 'RSM', 'SST']

✓ Turbulence Model Comparison table added to Excel file
✓ Format: Model: Mean (COV%) with percent difference from SST
✓ Professional formatting applied to comparison sheet

Sample comparison:

AoA_5:
  Lift (N):  SST: 68.111 N (10.11%) | RNG: 66.930 N (0.00%, -1.73% vs SST) | RSM: 64.663 N (0.00%, -5.06% vs SST)
  Drag (N):  SST: 1.429 N (53.21%) | RNG: 2.061 N (0.00%, +44.21% vs SST) | RSM: 1.943 N (0.00%, +35.97% vs SST)

AoA_10:
  Lift (N):  SST: 105.489 N (6.80%) | RNG: 108.209 N (0.00%, +2.58% vs SST) | RSM: 102.377 N (0.00%, -2.95% vs SST)
  Drag (N):  SST: 3.299 N (13.71%) | RNG: 3.850 N (0.00%, +16.70% vs SST) | RSM: 3.670 N (0.00%, +11.23% vs SST)

AoA_15:
  Lift (N):  SST: 79.078 N (17.87%) | RNG: 91.738 N (10.25%, +16.01% vs SST) | RSM: 84.395 N (10.96%, +6.72% vs SST)
  Drag (N):  SST: 16.558 N (19.64%) | RNG: 8.302 N (8.37%, -49.86% vs SST) | RSM: 15.306 N (14.22%, -7.56% vs SST)

AoA_20:
  Lift (N):  SST: 83.045 N (18

## Convergence Analysis (Optional)

In [59]:
# ==================== CONVERGENCE ANALYSIS TOOL ====================
# Inspired by GUI_Plotter_v12.txt methodology
# Tests different amounts of initial data removal to find optimal convergence window

def analyze_convergence(data_array, min_trim=0, max_trim=0.5, num_tests=10):
    """
    Analyze how statistics change when trimming different amounts of initial data.
    
    Parameters:
    - data_array: numpy array of lift or drag values
    - min_trim: minimum fraction of data to trim from beginning (0 to 1)
    - max_trim: maximum fraction of data to trim from beginning (0 to 1)
    - num_tests: number of trim amounts to test
    
    Returns:
    - Dictionary with trim fractions, means, std devs, and COVs
    """
    total_iterations = len(data_array)
    trim_fractions = np.linspace(min_trim, max_trim, num_tests)
    
    results = {
        'iterations_removed': [],
        'iterations_used': [],
        'mean': [],
        'std_dev': [],
        'cov': []
    }
    
    for trim_frac in trim_fractions:
        iterations_to_remove = int(total_iterations * trim_frac)
        trimmed_data = data_array[iterations_to_remove:]
        
        if len(trimmed_data) > 0:
            mean_val = np.mean(trimmed_data)
            std_val = np.std(trimmed_data)
            cov_val = (std_val / mean_val * 100) if mean_val != 0 else 0
            
            results['iterations_removed'].append(iterations_to_remove)
            results['iterations_used'].append(len(trimmed_data))
            results['mean'].append(mean_val)
            results['std_dev'].append(std_val)
            results['cov'].append(cov_val)
    
    return results

def plot_convergence_analysis(config, aoa, lift_data, drag_data, output_dir):
    """
    Create convergence analysis plots showing how statistics change with data trimming.
    """
    # Analyze both lift and drag
    lift_results = analyze_convergence(np.array(lift_data), min_trim=0, max_trim=0.7, num_tests=15)
    drag_results = analyze_convergence(np.array(drag_data), min_trim=0, max_trim=0.7, num_tests=15)
    
    # Create figure with subplots
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
    
    # Plot 1: Lift Mean vs Iterations Removed
    ax1.plot(lift_results['iterations_removed'], lift_results['mean'], 'o-', linewidth=2, markersize=8, color='#1f77b4')
    ax1.set_xlabel('Iterations Removed from Start')
    ax1.set_ylabel('Lift Mean (N)')
    ax1.set_title(f'Lift Mean Convergence\n{config} - {aoa}', fontweight='bold')
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Lift COV vs Iterations Removed
    ax2.plot(lift_results['iterations_removed'], lift_results['cov'], 'o-', linewidth=2, markersize=8, color='#ff7f0e')
    ax2.set_xlabel('Iterations Removed from Start')
    ax2.set_ylabel('Lift COV (%)')
    ax2.set_title(f'Lift COV Convergence\n{config} - {aoa}', fontweight='bold')
    ax2.grid(True, alpha=0.3)
    
    # Highlight minimum COV point for lift
    min_cov_idx = np.argmin(lift_results['cov'])
    ax2.axvline(x=lift_results['iterations_removed'][min_cov_idx], color='red', linestyle='--', linewidth=2, alpha=0.7)
    ax2.text(lift_results['iterations_removed'][min_cov_idx], max(lift_results['cov']), 
             f"  Min COV\n  Remove: {lift_results['iterations_removed'][min_cov_idx]}\n  Use: {lift_results['iterations_used'][min_cov_idx]}", 
             color='red', fontweight='bold', fontsize=9)
    
    # Plot 3: Drag Mean vs Iterations Removed
    ax3.plot(drag_results['iterations_removed'], drag_results['mean'], 'o-', linewidth=2, markersize=8, color='#2ca02c')
    ax3.set_xlabel('Iterations Removed from Start')
    ax3.set_ylabel('Drag Mean (N)')
    ax3.set_title(f'Drag Mean Convergence\n{config} - {aoa}', fontweight='bold')
    ax3.grid(True, alpha=0.3)
    
    # Plot 4: Drag COV vs Iterations Removed
    ax4.plot(drag_results['iterations_removed'], drag_results['cov'], 'o-', linewidth=2, markersize=8, color='#d62728')
    ax4.set_xlabel('Iterations Removed from Start')
    ax4.set_ylabel('Drag COV (%)')
    ax4.set_title(f'Drag COV Convergence\n{config} - {aoa}', fontweight='bold')
    ax4.grid(True, alpha=0.3)
    
    # Highlight minimum COV point for drag
    min_cov_idx = np.argmin(drag_results['cov'])
    ax4.axvline(x=drag_results['iterations_removed'][min_cov_idx], color='red', linestyle='--', linewidth=2, alpha=0.7)
    ax4.text(drag_results['iterations_removed'][min_cov_idx], max(drag_results['cov']), 
             f"  Min COV\n  Remove: {drag_results['iterations_removed'][min_cov_idx]}\n  Use: {drag_results['iterations_used'][min_cov_idx]}", 
             color='red', fontweight='bold', fontsize=9)
    
    plt.tight_layout()
    
    # Save convergence analysis plot
    convergence_dir = os.path.join(output_dir, "convergence_analysis")
    os.makedirs(convergence_dir, exist_ok=True)
    
    plot_file = os.path.join(convergence_dir, f"convergence_{config}.png")
    plt.savefig(plot_file, dpi=300, bbox_inches='tight')
    plt.close()
    
    return lift_results, drag_results, plot_file

# ==================== USER INPUT ====================
# Set to True to run convergence analysis, False to skip
RUN_CONVERGENCE_ANALYSIS = True

if RUN_CONVERGENCE_ANALYSIS:
    print("\n" + "=" * 80)
    print("RUNNING CONVERGENCE ANALYSIS")
    print("=" * 80)
    
    # Create convergence analysis directory
    convergence_dir = os.path.join(output_dir, "convergence_analysis")
    os.makedirs(convergence_dir, exist_ok=True)
    
    # Run convergence analysis on all configurations
    convergence_results = {}
    
    # Create text file for convergence data
    convergence_text_file = os.path.join(convergence_dir, "Convergence_Analysis_Results.txt")
    
    with open(convergence_text_file, 'w') as f:
        f.write("=" * 120 + "\n")
        f.write("CONVERGENCE ANALYSIS RESULTS\n")
        f.write("=" * 120 + "\n\n")
        
        for (config, aoa), data in all_data.items():
            print(f"\nAnalyzing: {config} - {aoa}")
            
            lift_results, drag_results, plot_path = plot_convergence_analysis(
                config, aoa,
                data['lift'],
                data['drag'],
                output_dir
            )
            
            convergence_results[(config, aoa)] = {
                'lift': lift_results,
                'drag': drag_results,
                'plot': plot_path
            }
            
            # Print optimization recommendations
            lift_min_cov_idx = np.argmin(lift_results['cov'])
            drag_min_cov_idx = np.argmin(drag_results['cov'])
            
            print(f"  ✓ Plot saved: {plot_path}")
            print(f"  ✓ Lift - Min COV at {lift_results['iterations_removed'][lift_min_cov_idx]} iterations removed (use {lift_results['iterations_used'][lift_min_cov_idx]})")
            print(f"  ✓ Drag - Min COV at {drag_results['iterations_removed'][drag_min_cov_idx]} iterations removed (use {drag_results['iterations_used'][drag_min_cov_idx]})")
            
            # Write to text file
            f.write(f"Configuration: {config}\n")
            f.write(f"Angle of Attack: {aoa}\n")
            f.write(f"Total Iterations: {len(data['lift'])}\n")
            f.write("-" * 120 + "\n\n")
            
            # Write LIFT table
            f.write("LIFT CONVERGENCE:\n")
            f.write(f"{'Iterations_Removed':<20} {'Iterations_Used':<20} {'Mean':<15} {'StdDev':<15} {'COV(%)':<10}\n")
            f.write("-" * 120 + "\n")
            for i in range(len(lift_results['iterations_removed'])):
                marker = " <-- MIN COV" if i == lift_min_cov_idx else ""
                f.write(f"{lift_results['iterations_removed'][i]:<20} "
                       f"{lift_results['iterations_used'][i]:<20} "
                       f"{lift_results['mean'][i]:<15.6f} "
                       f"{lift_results['std_dev'][i]:<15.6f} "
                       f"{lift_results['cov'][i]:<10.2f}{marker}\n")
            
            f.write("\n")
            
            # Write DRAG table
            f.write("DRAG CONVERGENCE:\n")
            f.write(f"{'Iterations_Removed':<20} {'Iterations_Used':<20} {'Mean':<15} {'StdDev':<15} {'COV(%)':<10}\n")
            f.write("-" * 120 + "\n")
            for i in range(len(drag_results['iterations_removed'])):
                marker = " <-- MIN COV" if i == drag_min_cov_idx else ""
                f.write(f"{drag_results['iterations_removed'][i]:<20} "
                       f"{drag_results['iterations_used'][i]:<20} "
                       f"{drag_results['mean'][i]:<15.6f} "
                       f"{drag_results['std_dev'][i]:<15.6f} "
                       f"{drag_results['cov'][i]:<10.2f}{marker}\n")
            
            f.write("\n" + "=" * 120 + "\n\n")
    
    print("\n" + "=" * 80)
    print("CONVERGENCE ANALYSIS COMPLETE")
    print(f"✓ Total configurations analyzed: {len(convergence_results)}")
    print(f"✓ Plots saved to: {os.path.join(output_dir, 'convergence_analysis')}")
    print(f"✓ Text file saved: {convergence_text_file}")
    print("=" * 80)
    
    # ==================== OPTIMIZED STATISTICS ====================
    # Apply convergence findings to recalculate statistics with optimal trim points
    print("\n" + "=" * 80)
    print("APPLYING OPTIMIZED CONVERGENCE SETTINGS")
    print("=" * 80)
    
    optimized_summary = []
    
    for (config, aoa), data in all_data.items():
        if (config, aoa) in convergence_results:
            conv = convergence_results[(config, aoa)]
            
            # Find optimal trim points (minimum COV)
            lift_min_cov_idx = np.argmin(conv['lift']['cov'])
            drag_min_cov_idx = np.argmin(conv['drag']['cov'])
            
            optimal_lift_trim = conv['lift']['iterations_removed'][lift_min_cov_idx]
            optimal_drag_trim = conv['drag']['iterations_removed'][drag_min_cov_idx]
            
            # Use the more conservative (higher) trim value for both
            optimal_trim = max(optimal_lift_trim, optimal_drag_trim)
            
            # Apply optimal trim
            lift_optimized = data['lift'][optimal_trim:]
            drag_optimized = data['drag'][optimal_trim:]
            
            # Calculate optimized statistics
            lift_mean_opt = np.mean(lift_optimized) if lift_optimized else 0
            lift_std_opt = np.std(lift_optimized) if lift_optimized else 0
            lift_cov_opt = (lift_std_opt / lift_mean_opt * 100) if lift_mean_opt != 0 else 0
            
            drag_mean_opt = np.mean(drag_optimized) if drag_optimized else 0
            drag_std_opt = np.std(drag_optimized) if drag_optimized else 0
            drag_cov_opt = (drag_std_opt / drag_mean_opt * 100) if drag_mean_opt != 0 else 0
            
            # Compare with original fixed-iteration statistics
            lift_original = data['lift'][-num_iterations:] if len(data['lift']) >= num_iterations else data['lift']
            drag_original = data['drag'][-num_iterations:] if len(data['drag']) >= num_iterations else data['drag']
            
            lift_mean_orig = np.mean(lift_original) if lift_original else 0
            lift_cov_orig = (np.std(lift_original) / lift_mean_orig * 100) if lift_mean_orig != 0 else 0
            
            drag_mean_orig = np.mean(drag_original) if drag_original else 0
            drag_cov_orig = (np.std(drag_original) / drag_mean_orig * 100) if drag_mean_orig != 0 else 0
            
            # Calculate improvements
            lift_cov_improvement = lift_cov_orig - lift_cov_opt
            drag_cov_improvement = drag_cov_orig - drag_cov_opt
            
            optimized_summary.append({
                'Configuration': config,
                'AoA': aoa,
                'Turbulence Model': data['turbulence_model'],
                'Original Iterations': len(lift_original),
                'Optimal Trim': optimal_trim,
                'Optimized Iterations': len(lift_optimized),
                'Lift Mean (Orig)': f"{lift_mean_orig:.3f}",
                'Lift Mean (Opt)': f"{lift_mean_opt:.3f}",
                'Lift COV (Orig)': f"{lift_cov_orig:.2f}%",
                'Lift COV (Opt)': f"{lift_cov_opt:.2f}%",
                'Lift COV Δ': f"{lift_cov_improvement:+.2f}%",
                'Drag Mean (Orig)': f"{drag_mean_orig:.3f}",
                'Drag Mean (Opt)': f"{drag_mean_opt:.3f}",
                'Drag COV (Orig)': f"{drag_cov_orig:.2f}%",
                'Drag COV (Opt)': f"{drag_cov_opt:.2f}%",
                'Drag COV Δ': f"{drag_cov_improvement:+.2f}%"
            })
            
            print(f"\n{config} - {aoa}:")
            print(f"  Optimal trim: {optimal_trim} iterations")
            print(f"  Lift COV: {lift_cov_orig:.2f}% → {lift_cov_opt:.2f}% (Δ {lift_cov_improvement:+.2f}%)")
            print(f"  Drag COV: {drag_cov_orig:.2f}% → {drag_cov_opt:.2f}% (Δ {drag_cov_improvement:+.2f}%)")
    
    # Save optimized statistics to Excel
    optimized_df = pd.DataFrame(optimized_summary)
    with pd.ExcelWriter(excel_file, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
        optimized_df.to_excel(writer, sheet_name='Optimized_Statistics', index=False)
    
    # Apply formatting
    import time
    time.sleep(0.5)
    wb = load_workbook(excel_file)
    ws = wb['Optimized_Statistics']
    
    # Apply same professional formatting
    for column in ws.columns:
        max_length = 0
        column_letter = get_column_letter(column[0].column)
        for cell in column:
            try:
                if len(str(cell.value)) > max_length:
                    max_length = len(str(cell.value))
            except:
                pass
        adjusted_width = min(max_length + 2, 25)
        ws.column_dimensions[column_letter].width = adjusted_width
    
    for cell in ws[1]:
        cell.font = header_font
        cell.fill = header_fill
        cell.alignment = header_alignment
        cell.border = border_style
    
    for row_idx, row in enumerate(ws.iter_rows(min_row=2), start=2):
        fill_color = row_fill_light if row_idx % 2 == 0 else row_fill_white
        for cell in row:
            cell.alignment = data_alignment
            cell.border = border_style
            cell.fill = fill_color
    
    ws.row_dimensions[1].height = 30
    wb.save(excel_file)
    wb.close()
    
    print("\n" + "=" * 80)
    print("OPTIMIZATION COMPLETE")
    print(f"✓ Optimized statistics saved to Excel sheet: 'Optimized_Statistics'")
    print(f"✓ Average COV improvements calculated for all configurations")
    print("=" * 80)
    
else:
    print("\n✓ Convergence analysis skipped (set RUN_CONVERGENCE_ANALYSIS = True to enable)")


RUNNING CONVERGENCE ANALYSIS

Analyzing: 4.3.1.2.10 - AoA_10
  ✓ Plot saved: C:\Users\lukek\OneDrive - Liberty University\Group-F.L.U.I.D. Research - GRID-FINS - GRID-FINS\Test\2414_006_004.3\convergence_analysis\convergence_4.3.1.2.10.png
  ✓ Lift - Min COV at 840 iterations removed (use 361)
  ✓ Drag - Min COV at 840 iterations removed (use 361)

Analyzing: 4.3.1.2.15 - AoA_15
  ✓ Plot saved: C:\Users\lukek\OneDrive - Liberty University\Group-F.L.U.I.D. Research - GRID-FINS - GRID-FINS\Test\2414_006_004.3\convergence_analysis\convergence_4.3.1.2.15.png
  ✓ Lift - Min COV at 840 iterations removed (use 361)
  ✓ Drag - Min COV at 780 iterations removed (use 421)

Analyzing: 4.3.1.2.20 - AoA_20
  ✓ Plot saved: C:\Users\lukek\OneDrive - Liberty University\Group-F.L.U.I.D. Research - GRID-FINS - GRID-FINS\Test\2414_006_004.3\convergence_analysis\convergence_4.3.1.2.20.png
  ✓ Lift - Min COV at 840 iterations removed (use 361)
  ✓ Drag - Min COV at 840 iterations removed (use 361)

Analyz

## Coefficient Calculations and AoA Graphs

In [60]:
# ==================== CALCULATE COEFFICIENTS ====================
# Calculate C_L and C_D for each configuration

coefficient_data = {}

for (config, aoa), data in all_data.items():
    # Get optimized data if convergence analysis was run, otherwise use fixed iterations
    if RUN_CONVERGENCE_ANALYSIS and (config, aoa) in convergence_results:
        conv = convergence_results[(config, aoa)]
        lift_min_cov_idx = np.argmin(conv['lift']['cov'])
        drag_min_cov_idx = np.argmin(conv['drag']['cov'])
        
        optimal_lift_trim = conv['lift']['iterations_removed'][lift_min_cov_idx]
        optimal_drag_trim = conv['drag']['iterations_removed'][drag_min_cov_idx]
        optimal_trim = max(optimal_lift_trim, optimal_drag_trim)
        
        lift_values = data['lift'][optimal_trim:]
        drag_values = data['drag'][optimal_trim:]
    else:
        # Use fixed iterations approach
        lift_values = data['lift'][-num_iterations:] if len(data['lift']) >= num_iterations else data['lift']
        drag_values = data['drag'][-num_iterations:] if len(data['drag']) >= num_iterations else data['drag']
    
    # Calculate mean forces
    lift_mean = np.mean(lift_values) if lift_values else 0
    drag_mean = np.mean(drag_values) if drag_values else 0
    
    # Calculate coefficients
    C_L = lift_mean / Q_times_A if Q_times_A != 0 else 0
    C_D = drag_mean / Q_times_A if Q_times_A != 0 else 0
    
    # Calculate standard deviations and COV for coefficients
    lift_std = np.std(lift_values) if lift_values else 0
    drag_std = np.std(drag_values) if drag_values else 0
    
    C_L_std = lift_std / Q_times_A if Q_times_A != 0 else 0
    C_D_std = drag_std / Q_times_A if Q_times_A != 0 else 0
    
    C_L_cov = (C_L_std / C_L * 100) if C_L != 0 else 0
    C_D_cov = (C_D_std / C_D * 100) if C_D != 0 else 0
    
    # Extract AoA number
    aoa_degrees = extract_aoa_number(aoa)
    
    # Store coefficient data
    coefficient_data[(config, aoa)] = {
        'aoa_degrees': aoa_degrees,
        'C_L': C_L,
        'C_D': C_D,
        'C_L_std': C_L_std,
        'C_D_std': C_D_std,
        'C_L_cov': C_L_cov,
        'C_D_cov': C_D_cov,
        'turbulence_model': data['turbulence_model'],
        'version': data['version'],
        'geometry': data['geometry'],
        'mesh': data['mesh']
    }

print(f"\n✓ Coefficients calculated for {len(coefficient_data)} configurations")

# Debug: Show what versions and turbulence models are in the data
if coefficient_data:
    sample_key = list(coefficient_data.keys())[0]
    print(f"Debug - Sample data: {coefficient_data[sample_key]}")
    unique_versions = set(v['version'] for v in coefficient_data.values())
    unique_models = set(v['turbulence_model'] for v in coefficient_data.values())
    print(f"Debug - Available versions: {sorted(unique_versions)}")
    print(f"Debug - Available turbulence models: {sorted(unique_models)}")

# Filter coefficient data by selected turbulence model and version
if COEFF_TURBULENCE_MODEL == 'ALL' and COEFF_VERSION == 'ALL':
    filtered_coeff_data = coefficient_data
    print(f"✓ Showing all turbulence models and all versions in coefficient analysis")
elif COEFF_TURBULENCE_MODEL == 'ALL':
    filtered_coeff_data = {k: v for k, v in coefficient_data.items() if v['version'] == COEFF_VERSION}
    print(f"✓ Showing all turbulence models for version: {COEFF_VERSION}")
    print(f"✓ {len(filtered_coeff_data)} configurations match this version")
elif COEFF_VERSION == 'ALL':
    filtered_coeff_data = {k: v for k, v in coefficient_data.items() if v['turbulence_model'] == COEFF_TURBULENCE_MODEL}
    print(f"✓ Filtering to show only turbulence model: {COEFF_TURBULENCE_MODEL}")
    print(f"✓ {len(filtered_coeff_data)} configurations match this turbulence model")
else:
    filtered_coeff_data = {k: v for k, v in coefficient_data.items() 
                          if v['turbulence_model'] == COEFF_TURBULENCE_MODEL and v['version'] == COEFF_VERSION}
    print(f"✓ Filtering to show only: {COEFF_TURBULENCE_MODEL}, {COEFF_VERSION}")
    print(f"✓ {len(filtered_coeff_data)} configurations match these criteria")

if len(filtered_coeff_data) == 0:
    print(f"WARNING: No configurations match the filter criteria!")
    print(f"  - COEFF_TURBULENCE_MODEL is set to: '{COEFF_TURBULENCE_MODEL}'")
    print(f"  - COEFF_VERSION is set to: '{COEFF_VERSION}'")
    print(f"  - Try setting both to 'ALL' to see all data")

# Create summary DataFrame (filtered)
coeff_summary = []
for (config, aoa), coeff in sorted(filtered_coeff_data.items(), key=lambda x: (x[1]['aoa_degrees'], x[0][0])):
    coeff_summary.append({
        'Configuration': config,
        'AoA': aoa,
        'AoA (degrees)': coeff['aoa_degrees'],
        'Turbulence Model': coeff['turbulence_model'],
        'Version': coeff['version'],
        'C_L': f"{coeff['C_L']:.6f}",
        'C_L_std': f"{coeff['C_L_std']:.6f}",
        'C_L_COV (%)': f"{coeff['C_L_cov']:.2f}",
        'C_D': f"{coeff['C_D']:.6f}",
        'C_D_std': f"{coeff['C_D_std']:.6f}",
        'C_D_COV (%)': f"{coeff['C_D_cov']:.2f}"
    })

coeff_df = pd.DataFrame(coeff_summary)

# Save to Excel
with pd.ExcelWriter(excel_file, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    coeff_df.to_excel(writer, sheet_name='Coefficients', index=False)

# Apply formatting
import time
time.sleep(0.5)
wb = load_workbook(excel_file)
ws = wb['Coefficients']

for column in ws.columns:
    max_length = 0
    column_letter = get_column_letter(column[0].column)
    for cell in column:
        try:
            if len(str(cell.value)) > max_length:
                max_length = len(str(cell.value))
        except:
            pass
    adjusted_width = min(max_length + 2, 25)
    ws.column_dimensions[column_letter].width = adjusted_width

for cell in ws[1]:
    cell.font = header_font
    cell.fill = header_fill
    cell.alignment = header_alignment
    cell.border = border_style

for row_idx, row in enumerate(ws.iter_rows(min_row=2), start=2):
    fill_color = row_fill_light if row_idx % 2 == 0 else row_fill_white
    for cell in row:
        cell.alignment = data_alignment
        cell.border = border_style
        cell.fill = fill_color

ws.row_dimensions[1].height = 30
wb.save(excel_file)
wb.close()

print(f"✓ Coefficient data saved to Excel sheet: 'Coefficients'")


✓ Coefficients calculated for 19 configurations
Debug - Sample data: {'aoa_degrees': 10, 'C_L': np.float64(3.201651305556193), 'C_D': np.float64(0.11996805043869492), 'C_L_std': np.float64(1.0439789846559373e-07), 'C_D_std': np.float64(1.840816622874187e-08), 'C_L_cov': np.float64(3.2607516716270783e-06), 'C_D_cov': np.float64(1.5344223867460998e-05), 'turbulence_model': 'SST', 'version': 'Version 2', 'geometry': 'Baseline', 'mesh': 'Standard'}
Debug - Available versions: ['Version 1', 'Version 2', 'Version 3', 'Version 4']
Debug - Available turbulence models: ['RNG', 'RSM', 'SST']
✓ Filtering to show only: SST, Version 2
✓ 4 configurations match these criteria
✓ Coefficient data saved to Excel sheet: 'Coefficients'


In [61]:
# ==================== CREATE C_L vs AoA and C_D vs AoA GRAPHS ====================

# Create directory for coefficient graphs
coeff_graphs_dir = os.path.join(output_dir, "coefficient_graphs")
os.makedirs(coeff_graphs_dir, exist_ok=True)

# Organize data by turbulence model for plotting (use filtered data)
turbulence_coeff_data = defaultdict(lambda: {'aoa': [], 'C_L': [], 'C_D': [], 'C_L_std': [], 'C_D_std': []})

for (config, aoa), coeff in filtered_coeff_data.items():
    model = coeff['turbulence_model']
    turbulence_coeff_data[model]['aoa'].append(coeff['aoa_degrees'])
    turbulence_coeff_data[model]['C_L'].append(coeff['C_L'])
    turbulence_coeff_data[model]['C_D'].append(coeff['C_D'])
    turbulence_coeff_data[model]['C_L_std'].append(coeff['C_L_std'])
    turbulence_coeff_data[model]['C_D_std'].append(coeff['C_D_std'])

# Sort by AoA for each model
for model in turbulence_coeff_data:
    # Create array of tuples and sort
    combined = list(zip(
        turbulence_coeff_data[model]['aoa'],
        turbulence_coeff_data[model]['C_L'],
        turbulence_coeff_data[model]['C_D'],
        turbulence_coeff_data[model]['C_L_std'],
        turbulence_coeff_data[model]['C_D_std']
    ))
    combined.sort(key=lambda x: x[0])  # Sort by AoA
    
    # Unpack back to separate lists
    turbulence_coeff_data[model]['aoa'] = [x[0] for x in combined]
    turbulence_coeff_data[model]['C_L'] = [x[1] for x in combined]
    turbulence_coeff_data[model]['C_D'] = [x[2] for x in combined]
    turbulence_coeff_data[model]['C_L_std'] = [x[3] for x in combined]
    turbulence_coeff_data[model]['C_D_std'] = [x[4] for x in combined]

# Define colors and markers for different turbulence models
model_styles = {
    'SST': {'color': '#1f77b4', 'marker': 'o', 'label': 'SST'},
    'RNG': {'color': '#ff7f0e', 'marker': 's', 'label': 'RNG k-ε'},
    'RSM': {'color': '#2ca02c', 'marker': '^', 'label': 'RSM'},
    'k-epsilon': {'color': '#d62728', 'marker': 'D', 'label': 'k-ε'}
}

# ==================== PLOT 1: C_L vs AoA ====================
fig, ax = plt.subplots(figsize=(12, 8))

for model in sorted(turbulence_coeff_data.keys()):
    style = model_styles.get(model, {'color': 'gray', 'marker': 'o', 'label': model})
    
    aoa_vals = np.array(turbulence_coeff_data[model]['aoa'])
    C_L_vals = np.array(turbulence_coeff_data[model]['C_L'])
    C_L_std_vals = np.array(turbulence_coeff_data[model]['C_L_std'])
    
    # Plot with error bars
    ax.errorbar(aoa_vals, C_L_vals, yerr=C_L_std_vals,
                marker=style['marker'], markersize=8, linewidth=2, capsize=5,
                color=style['color'], label=style['label'], alpha=0.8)

ax.set_xlabel('Angle of Attack (degrees)', fontsize=14, fontweight='bold')
ax.set_ylabel('Lift Coefficient ($C_L$)', fontsize=14, fontweight='bold')
ax.set_title('Lift Coefficient vs Angle of Attack\nComparison of Turbulence Models', fontsize=16, fontweight='bold')
ax.grid(True, alpha=0.3, linestyle='--')
ax.legend(fontsize=12, loc='best', framealpha=0.9)
ax.tick_params(labelsize=11)

plt.tight_layout()
C_L_graph_file = os.path.join(coeff_graphs_dir, "C_L_vs_AoA.png")
plt.savefig(C_L_graph_file, dpi=300, bbox_inches='tight')
plt.close()

print(f"✓ C_L vs AoA graph saved: {C_L_graph_file}")

# ==================== PLOT 2: C_D vs AoA ====================
fig, ax = plt.subplots(figsize=(12, 8))

for model in sorted(turbulence_coeff_data.keys()):
    style = model_styles.get(model, {'color': 'gray', 'marker': 'o', 'label': model})
    
    aoa_vals = np.array(turbulence_coeff_data[model]['aoa'])
    C_D_vals = np.array(turbulence_coeff_data[model]['C_D'])
    C_D_std_vals = np.array(turbulence_coeff_data[model]['C_D_std'])
    
    # Plot with error bars
    ax.errorbar(aoa_vals, C_D_vals, yerr=C_D_std_vals,
                marker=style['marker'], markersize=8, linewidth=2, capsize=5,
                color=style['color'], label=style['label'], alpha=0.8)

ax.set_xlabel('Angle of Attack (degrees)', fontsize=14, fontweight='bold')
ax.set_ylabel('Drag Coefficient ($C_D$)', fontsize=14, fontweight='bold')
ax.set_title('Drag Coefficient vs Angle of Attack\nComparison of Turbulence Models', fontsize=16, fontweight='bold')
ax.grid(True, alpha=0.3, linestyle='--')
ax.legend(fontsize=12, loc='best', framealpha=0.9)
ax.tick_params(labelsize=11)

plt.tight_layout()
C_D_graph_file = os.path.join(coeff_graphs_dir, "C_D_vs_AoA.png")
plt.savefig(C_D_graph_file, dpi=300, bbox_inches='tight')
plt.close()

print(f"✓ C_D vs AoA graph saved: {C_D_graph_file}")

# ==================== PLOT 3: C_L vs C_D (Drag Polar) ====================
fig, ax = plt.subplots(figsize=(12, 8))

for model in sorted(turbulence_coeff_data.keys()):
    style = model_styles.get(model, {'color': 'gray', 'marker': 'o', 'label': model})
    
    C_L_vals = np.array(turbulence_coeff_data[model]['C_L'])
    C_D_vals = np.array(turbulence_coeff_data[model]['C_D'])
    C_L_std_vals = np.array(turbulence_coeff_data[model]['C_L_std'])
    C_D_std_vals = np.array(turbulence_coeff_data[model]['C_D_std'])
    
    # Plot with error bars
    ax.errorbar(C_D_vals, C_L_vals, xerr=C_D_std_vals, yerr=C_L_std_vals,
                marker=style['marker'], markersize=8, linewidth=2, capsize=5,
                color=style['color'], label=style['label'], alpha=0.8)
    
    # Add AoA annotations at each point
    for i, aoa in enumerate(turbulence_coeff_data[model]['aoa']):
        ax.annotate(f"{int(aoa)}°", 
                   (C_D_vals[i], C_L_vals[i]),
                   textcoords="offset points", xytext=(5, 5),
                   fontsize=8, alpha=0.7, color=style['color'])

ax.set_xlabel('Drag Coefficient ($C_D$)', fontsize=14, fontweight='bold')
ax.set_ylabel('Lift Coefficient ($C_L$)', fontsize=14, fontweight='bold')
ax.set_title('Drag Polar ($C_L$ vs $C_D$)\nComparison of Turbulence Models', fontsize=16, fontweight='bold')
ax.grid(True, alpha=0.3, linestyle='--')
ax.legend(fontsize=12, loc='best', framealpha=0.9)
ax.tick_params(labelsize=11)

plt.tight_layout()
polar_graph_file = os.path.join(coeff_graphs_dir, "Drag_Polar.png")
plt.savefig(polar_graph_file, dpi=300, bbox_inches='tight')
plt.close()

print(f"✓ Drag polar graph saved: {polar_graph_file}")

# ==================== PLOT 4: Combined C_L and C_D vs AoA ====================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))

# Left plot: C_L vs AoA
for model in sorted(turbulence_coeff_data.keys()):
    style = model_styles.get(model, {'color': 'gray', 'marker': 'o', 'label': model})
    
    aoa_vals = np.array(turbulence_coeff_data[model]['aoa'])
    C_L_vals = np.array(turbulence_coeff_data[model]['C_L'])
    C_L_std_vals = np.array(turbulence_coeff_data[model]['C_L_std'])
    
    ax1.errorbar(aoa_vals, C_L_vals, yerr=C_L_std_vals,
                marker=style['marker'], markersize=8, linewidth=2, capsize=5,
                color=style['color'], label=style['label'], alpha=0.8)

ax1.set_xlabel('Angle of Attack (degrees)', fontsize=14, fontweight='bold')
ax1.set_ylabel('Lift Coefficient ($C_L$)', fontsize=14, fontweight='bold')
ax1.set_title('Lift Coefficient vs AoA', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3, linestyle='--')
ax1.legend(fontsize=11, loc='best', framealpha=0.9)
ax1.tick_params(labelsize=11)

# Right plot: C_D vs AoA
for model in sorted(turbulence_coeff_data.keys()):
    style = model_styles.get(model, {'color': 'gray', 'marker': 'o', 'label': model})
    
    aoa_vals = np.array(turbulence_coeff_data[model]['aoa'])
    C_D_vals = np.array(turbulence_coeff_data[model]['C_D'])
    C_D_std_vals = np.array(turbulence_coeff_data[model]['C_D_std'])
    
    ax2.errorbar(aoa_vals, C_D_vals, yerr=C_D_std_vals,
                marker=style['marker'], markersize=8, linewidth=2, capsize=5,
                color=style['color'], label=style['label'], alpha=0.8)

ax2.set_xlabel('Angle of Attack (degrees)', fontsize=14, fontweight='bold')
ax2.set_ylabel('Drag Coefficient ($C_D$)', fontsize=14, fontweight='bold')
ax2.set_title('Drag Coefficient vs AoA', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3, linestyle='--')
ax2.legend(fontsize=11, loc='best', framealpha=0.9)
ax2.tick_params(labelsize=11)

plt.tight_layout()
combined_graph_file = os.path.join(coeff_graphs_dir, "C_L_C_D_Combined.png")
plt.savefig(combined_graph_file, dpi=300, bbox_inches='tight')
plt.close()

print(f"✓ Combined C_L and C_D graph saved: {combined_graph_file}")

print("\n" + "=" * 80)
print("COEFFICIENT GRAPHS COMPLETE")
print(f"✓ All graphs saved to: {coeff_graphs_dir}")
print("  - C_L_vs_AoA.png")
print("  - C_D_vs_AoA.png")
print("  - Drag_Polar.png")
print("  - C_L_C_D_Combined.png")
print("=" * 80)

✓ C_L vs AoA graph saved: C:\Users\lukek\OneDrive - Liberty University\Group-F.L.U.I.D. Research - GRID-FINS - GRID-FINS\Test\2414_006_004.3\coefficient_graphs\C_L_vs_AoA.png
✓ C_D vs AoA graph saved: C:\Users\lukek\OneDrive - Liberty University\Group-F.L.U.I.D. Research - GRID-FINS - GRID-FINS\Test\2414_006_004.3\coefficient_graphs\C_D_vs_AoA.png
✓ Drag polar graph saved: C:\Users\lukek\OneDrive - Liberty University\Group-F.L.U.I.D. Research - GRID-FINS - GRID-FINS\Test\2414_006_004.3\coefficient_graphs\Drag_Polar.png
✓ Combined C_L and C_D graph saved: C:\Users\lukek\OneDrive - Liberty University\Group-F.L.U.I.D. Research - GRID-FINS - GRID-FINS\Test\2414_006_004.3\coefficient_graphs\C_L_C_D_Combined.png

COEFFICIENT GRAPHS COMPLETE
✓ All graphs saved to: C:\Users\lukek\OneDrive - Liberty University\Group-F.L.U.I.D. Research - GRID-FINS - GRID-FINS\Test\2414_006_004.3\coefficient_graphs
  - C_L_vs_AoA.png
  - C_D_vs_AoA.png
  - Drag_Polar.png
  - C_L_C_D_Combined.png
